# 03 — KPI Analysis & Business Investigation

**Project:** Olist Brazilian E-Commerce — Delivery Reliability & Customer Experience  
**Audience:** Operations manager / business lead  
**Stage:** Calculate defensible baseline KPIs, investigate delivery performance, and identify segments that warrant follow-up.

## Business question

**Where are delivery delays concentrated, and how are they associated with customer satisfaction?**

This notebook uses the validated analytical model created by Notebook 02. It deliberately separates:

1. **Measurement:** define the population and calculate consistent KPIs.
2. **Investigation:** compare monthly cohorts, customer states, categories, sellers, and fulfillment stages.
3. **Action:** identify priority areas to investigate, without claiming that an observed association proves a cause.

### KPI rules

- Delivery performance is measured among delivered orders with valid purchase, actual-delivery, and estimated-delivery calendar dates.
- **Late** means actual delivery calendar date is later than estimated delivery calendar date. Same-calendar-day delivery is not late.
- Review scores are averaged to one score per order first. Low-rated means the order-level average score is **2 or lower**.
- The state table requires at least 500 eligible delivered orders; the category table requires at least 100 order-category observations; the seller table requires at least 100 distinct order exposures. These are portfolio analysis thresholds, not universal statistical rules.
- A single order can contain multiple product categories or sellers. Therefore, category and seller tables show **order exposure**, and their row counts must not be summed to get unique marketplace orders.
- This is historical data, largely from 2016–2018. Recent purchase cohorts may have incomplete delivery follow-up. Segment differences are descriptive and do not prove causality.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "analytical_model"
ANALYSIS_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "business_analysis"
CHART_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "charts"
for folder in [ANALYSIS_OUTPUT_DIR, CHART_OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

ORDER_PATH = PROCESSED_DIR / "order_level_analytics.csv"
ITEM_PATH = PROCESSED_DIR / "order_item_analytics.csv"
VALIDATION_PATH = MODEL_OUTPUT_DIR / "model_validation_report.csv"

missing = [str(p) for p in [ORDER_PATH, ITEM_PATH, VALIDATION_PATH] if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Required model outputs are missing:\n- " + "\n- ".join(missing) +
        "\nRun Notebook 01 with EXPORT_CLEANED_TABLES=True, then run Notebook 02 with EXPORT_ANALYTIC_TABLES=True."
    )

validation_report = pd.read_csv(VALIDATION_PATH)
if "status" not in validation_report.columns or not validation_report["status"].astype("string").str.upper().eq("PASS").all():
    raise AssertionError("Notebook 02 model validation contains missing or non-PASS results. Resolve those before KPI analysis.")

print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {ANALYSIS_OUTPUT_DIR.relative_to(PROJECT_ROOT)}")
print(f"Model validation checks passing: {len(validation_report):,}")

## 1. Load models and validate the analysis population

We validate the model again at the point of use. A successful upstream notebook is necessary, but each downstream notebook should still check assumptions it relies on.

In [ ]:
ORDER_DATE_COLUMNS = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]
order_dtype = {
    "order_id": "string", "customer_id": "string", "customer_unique_id": "string",
    "order_status": "string", "customer_state": "string", "order_purchase_month": "string",
}
item_dtype = {
    "order_id": "string", "product_id": "string", "seller_id": "string",
    "customer_state": "string", "product_category_label": "string",
}
orders = pd.read_csv(ORDER_PATH, dtype=order_dtype, low_memory=False)
items = pd.read_csv(ITEM_PATH, dtype=item_dtype, low_memory=False)
for col in ORDER_DATE_COLUMNS:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors="coerce")

numeric_order_columns = [
    "average_review_score", "delivery_delay_days", "purchase_to_delivery_calendar_days",
    "purchase_to_approval_hours", "approval_to_carrier_hours", "carrier_to_delivery_hours",
    "item_price_total", "freight_value_total", "item_total_including_freight", "payment_value_total",
]
for col in numeric_order_columns:
    if col in orders.columns:
        orders[col] = pd.to_numeric(orders[col], errors="coerce")
for col in ["price", "freight_value", "item_total_including_freight"]:
    if col in items.columns:
        items[col] = pd.to_numeric(items[col], errors="coerce")


def as_boolean(series: pd.Series) -> pd.Series:
    """Parse booleans robustly from bool, nullable bool, or CSV strings."""
    if pd.api.types.is_bool_dtype(series.dtype):
        return series.fillna(False).astype(bool)
    normalized = series.astype("string").str.strip().str.lower()
    return normalized.map({"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False}).fillna(False).astype(bool)

for required_col in ["order_id", "order_status", "delivery_metric_eligible", "is_late_delivery", "average_review_score"]:
    if required_col not in orders.columns:
        raise KeyError(f"Expected column missing from order-level model: {required_col}")
for required_col in ["order_id", "order_item_id", "product_category_label", "seller_id"]:
    if required_col not in items.columns:
        raise KeyError(f"Expected column missing from order-item model: {required_col}")

orders["delivery_metric_eligible"] = as_boolean(orders["delivery_metric_eligible"])
orders["is_late_delivery"] = as_boolean(orders["is_late_delivery"])
orders["has_review_score"] = orders["average_review_score"].notna()
orders["is_low_rated_order_calc"] = orders["average_review_score"].le(2).where(orders["has_review_score"])

# Core grain and reconciliation checks.
assert orders["order_id"].notna().all() and orders["order_id"].is_unique, "Order model must have one row per order_id."
assert not items.duplicated(["order_id", "order_item_id"]).any(), "Order-item model has duplicate order_id + order_item_id rows."
assert orders["delivery_metric_eligible"].sum() == orders.loc[orders["delivery_metric_eligible"], "order_id"].nunique()
assert orders.loc[orders["delivery_metric_eligible"], "is_late_delivery"].notna().all(), "Eligible delivered orders must have a boolean late classification."

eligible = orders.loc[orders["delivery_metric_eligible"]].copy()
eligible["is_late_delivery"] = eligible["is_late_delivery"].astype(bool)
eligible["is_low_rated_order_calc"] = eligible["is_low_rated_order_calc"].astype("boolean")
reviewed_eligible = eligible.loc[eligible["has_review_score"]].copy()

print(f"Orders in model: {len(orders):,}")
print(f"Order items in model: {len(items):,}")
print(f"Eligible delivered orders: {len(eligible):,}")
print(f"Reviewed eligible delivered orders: {len(reviewed_eligible):,}")

## 2. Baseline KPIs

The executive KPI table includes each metric's value, definition, denominator/population, and interpretation. Keep the definition alongside the metric when you move it into Power BI.

In [ ]:
n_orders = int(orders["order_id"].nunique())
status = orders["order_status"].astype("string").str.lower()
n_delivered_status = int(status.eq("delivered").sum())
n_canceled = int(status.eq("canceled").sum())
n_unavailable = int(status.eq("unavailable").sum())
n_eligible = int(len(eligible))
n_late = int(eligible["is_late_delivery"].sum())
n_ontime = int((~eligible["is_late_delivery"]).sum())
late_rate = n_late / n_eligible if n_eligible else np.nan
on_time_rate = n_ontime / n_eligible if n_eligible else np.nan

late_group = reviewed_eligible.loc[reviewed_eligible["is_late_delivery"]]
ontime_group = reviewed_eligible.loc[~reviewed_eligible["is_late_delivery"]]
late_mean_score = late_group["average_review_score"].mean()
ontime_mean_score = ontime_group["average_review_score"].mean()
late_low_share = late_group["is_low_rated_order_calc"].mean()
ontime_low_share = ontime_group["is_low_rated_order_calc"].mean()

late_rows = eligible.loc[eligible["is_late_delivery"]]
late_delay = pd.to_numeric(late_rows.get("delivery_delay_days", pd.Series(dtype=float)), errors="coerce").dropna()
lead_time = pd.to_numeric(eligible.get("purchase_to_delivery_calendar_days", pd.Series(dtype=float)), errors="coerce").dropna()

# Item-value KPIs use orders with available item records; they are not profit measures.
if "item_price_total" in eligible.columns:
    delivered_with_item_value = eligible[eligible["item_price_total"].notna()]
    delivered_item_price_total = delivered_with_item_value["item_price_total"].sum(min_count=1)
    delivered_item_aov = delivered_item_price_total / delivered_with_item_value["order_id"].nunique() if len(delivered_with_item_value) else np.nan
else:
    delivered_with_item_value = pd.DataFrame()
    delivered_item_price_total = np.nan
    delivered_item_aov = np.nan

kpi_rows = [
    ("Orders placed", n_orders, "Count of distinct order IDs in the order model.", "All orders in the historical dataset.", "Overall order volume; not a count of completed sales."),
    ("Delivered-status orders", n_delivered_status, "Orders whose recorded status is delivered.", "All orders.", "Status count; a subset can fail delivery-metric eligibility because dates are missing or invalid."),
    ("Canceled-order rate", n_canceled / n_orders if n_orders else np.nan, "Canceled orders divided by all orders.", "All orders; status = canceled.", "Describes recorded cancellations, not the reason or preventability."),
    ("Unavailable-order rate", n_unavailable / n_orders if n_orders else np.nan, "Unavailable orders divided by all orders.", "All orders; status = unavailable.", "Keep separate from cancellations."),
    ("Eligible delivered orders", n_eligible, "Delivered orders with valid purchase, actual-delivery and estimated-delivery calendar dates.", "Delivery-performance population.", "Denominator for delivery KPIs."),
    ("Late delivered orders", n_late, "Eligible delivered orders where actual delivery date is after estimated delivery date.", "Eligible delivered orders.", "Number of delayed orders in the measured population."),
    ("Late delivery rate", late_rate, "Late eligible delivered orders divided by all eligible delivered orders.", f"{n_eligible:,} eligible delivered orders.", "Conditional on delivery being recorded; do not interpret as a late rate for all orders placed."),
    ("On-time / early delivery rate", on_time_rate, "Eligible delivered orders received on or before the estimated calendar date divided by eligible delivered orders.", f"{n_eligible:,} eligible delivered orders.", "Complements the late delivery rate."),
    ("Median purchase-to-delivery days", lead_time.median(), "Median calendar days from purchase date to actual delivery date.", f"{len(lead_time):,} eligible orders with valid interval.", "Describes typical wait; different from lateness against the promise."),
    ("Median delay among late orders (days)", late_delay.median(), "Median actual delivery date minus estimated delivery date, restricted to positive delays.", f"{len(late_delay):,} late orders with delay values.", "Early deliveries do not offset the size of late delays."),
    ("Average order-level review score", orders.loc[orders["has_review_score"], "average_review_score"].mean(), "Mean of order-level average review scores.", f"{int(orders['has_review_score'].sum()):,} orders with one or more numeric review scores.", "Each order gets equal weight after its own reviews are averaged."),
    ("Low-rating share", orders.loc[orders["has_review_score"], "is_low_rated_order_calc"].mean(), "Orders with an order-level mean score <= 2, divided by orders with scores.", f"{int(orders['has_review_score'].sum()):,} reviewed orders.", "An explicit portfolio definition; document it in the dashboard."),
    ("Mean review score — late orders", late_mean_score, "Mean order-level review score among eligible late orders with a score.", f"{len(late_group):,} reviewed late orders.", "Descriptive association; not proof of causality."),
    ("Mean review score — on-time / early", ontime_mean_score, "Mean order-level review score among eligible on-time/early orders with a score.", f"{len(ontime_group):,} reviewed on-time/early orders.", "Compare using the same scoring method as late orders."),
    ("Review-score gap (on-time minus late)", ontime_mean_score - late_mean_score, "Mean on-time/early score minus mean late-order score.", "Reviewed eligible delivered orders, summarized at order level.", "A positive gap shows lower scores in the late group; unadjusted for other factors."),
    ("Low-rating share — late orders", late_low_share, "Share of reviewed late orders with order-level mean review score <= 2.", f"{len(late_group):,} reviewed late orders.", "Helps quantify customer experience among late deliveries."),
    ("Low-rating share — on-time / early", ontime_low_share, "Share of reviewed on-time/early orders with order-level mean review score <= 2.", f"{len(ontime_group):,} reviewed on-time/early orders.", "Comparison group for the low-rating share."),
    ("Delivered item-price total (BRL)", delivered_item_price_total, "Sum of item price totals among eligible delivered orders with item data; excludes freight.", f"{len(delivered_with_item_value):,} eligible delivered orders with item data.", "Merchandise value only; not profit and not net revenue."),
    ("Average item price per eligible delivered order (BRL)", delivered_item_aov, "Delivered item-price total divided by eligible delivered orders with item data.", f"{len(delivered_with_item_value):,} eligible delivered orders with item data.", "Excludes freight and is not an accounting AOV unless the definition is adopted."),
]

kpi_summary = pd.DataFrame(kpi_rows, columns=["kpi", "value", "definition", "population_denominator", "interpretation"])
percent_kpis = {
    "Canceled-order rate", "Unavailable-order rate", "Late delivery rate", "On-time / early delivery rate",
    "Low-rating share", "Low-rating share — late orders", "Low-rating share — on-time / early",
}
score_kpis = {"Average order-level review score", "Mean review score — late orders", "Mean review score — on-time / early"}
day_kpis = {"Median purchase-to-delivery days", "Median delay among late orders (days)"}
value_kpis = {"Delivered item-price total (BRL)", "Average item price per eligible delivered order (BRL)"}
point_kpis = {"Review-score gap (on-time minus late)"}

def kpi_unit_and_display(row):
    name, value = row["kpi"], row["value"]
    if pd.isna(value):
        return pd.Series({"value_unit": "unknown", "display_value": "N/A"})
    if name in percent_kpis:
        return pd.Series({"value_unit": "percent", "display_value": f"{value:.2%}"})
    if name in score_kpis:
        return pd.Series({"value_unit": "score out of 5", "display_value": f"{value:.2f} / 5"})
    if name in day_kpis:
        return pd.Series({"value_unit": "days", "display_value": f"{value:.1f} days"})
    if name in value_kpis:
        return pd.Series({"value_unit": "BRL", "display_value": f"R$ {value:,.2f}"})
    if name in point_kpis:
        return pd.Series({"value_unit": "score points", "display_value": f"{value:.2f} points"})
    return pd.Series({"value_unit": "orders", "display_value": f"{int(value):,}"})

kpi_summary[["value_unit", "display_value"]] = kpi_summary.apply(kpi_unit_and_display, axis=1)
kpi_summary.to_csv(ANALYSIS_OUTPUT_DIR / "executive_kpi_summary.csv", index=False)
display(kpi_summary)

## 3. Delivery outcomes and review comparison

This table makes group sample sizes visible and avoids comparing average review-record scores to per-order scores. The late and on-time/early groups are drawn from eligible delivered orders only.

In [ ]:
comparison_rows = []
for label, group in [("Late", late_group), ("On-time or early", ontime_group)]:
    n = len(group)
    comparison_rows.append({
        "delivery_group": label,
        "eligible_reviewed_orders": int(n),
        "mean_order_review_score": group["average_review_score"].mean(),
        "median_order_review_score": group["average_review_score"].median(),
        "low_rated_orders_score_le_2": int(group["is_low_rated_order_calc"].sum()),
        "low_rating_share": group["is_low_rated_order_calc"].mean(),
        "share_of_reviewed_eligible_orders": n / len(reviewed_eligible) if len(reviewed_eligible) else np.nan,
    })
review_comparison = pd.DataFrame(comparison_rows)
review_comparison.to_csv(ANALYSIS_OUTPUT_DIR / "delivery_review_comparison.csv", index=False)
display(review_comparison)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
plot_data = review_comparison.set_index("delivery_group")["mean_order_review_score"]
plot_data.plot(kind="bar", ax=ax)
ax.set_title("Average order-level review score by delivery outcome")
ax.set_xlabel("")
ax.set_ylabel("Mean review score (1–5)")
ax.set_ylim(0, 5)
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(CHART_OUTPUT_DIR / "review_score_by_delivery_outcome.png", dpi=160, bbox_inches="tight")
plt.show()

## 4. Monthly delivery performance

Orders are grouped by **purchase month** (cohort), not by delivery month. The late-delivery rate denominator remains eligible delivered orders in each cohort. First and last observed purchase months are flagged as partial calendar months. The notebook also detects a sharp volume collapse in the final six observed months and flags that tail as potentially under-covered; this is a warning heuristic, not proof. Finally, it estimates whether each cohort has at least the 99th-percentile observed purchase-to-delivery lead time of follow-up before the latest recorded delivery date. The trend chart also requires at least 500 eligible delivered orders in a month, so tiny denominators do not create misleading spikes. The trend chart excludes partial, low-volume-tail, insufficient-follow-up, and low-volume cohorts.

In [ ]:
monthly = orders.copy()
monthly["purchase_month"] = monthly["order_purchase_timestamp"].dt.to_period("M").astype("string")
monthly["is_late_eligible"] = monthly["delivery_metric_eligible"] & monthly["is_late_delivery"]
monthly["is_eligible_order_int"] = monthly["delivery_metric_eligible"].astype(int)
monthly["is_late_order_int"] = monthly["is_late_eligible"].astype(int)
monthly["review_score_for_month"] = monthly["average_review_score"].where(monthly["has_review_score"])

monthly = monthly.loc[monthly["order_purchase_timestamp"].notna()].copy()
monthly_perf = monthly.groupby("purchase_month", dropna=True).agg(
    orders_placed=("order_id", "nunique"),
    eligible_delivered_orders=("is_eligible_order_int", "sum"),
    late_orders=("is_late_order_int", "sum"),
    mean_order_review_score=("review_score_for_month", "mean"),
).reset_index()

valid_purchase_dates = orders["order_purchase_timestamp"].dropna()
first_purchase_date = valid_purchase_dates.min()
last_purchase_date = valid_purchase_dates.max()
first_month = first_purchase_date.to_period("M").strftime("%Y-%m")
last_month = last_purchase_date.to_period("M").strftime("%Y-%m")

# Reindex to every calendar month so months with zero records are visible instead of silently disappearing.
monthly_perf["_month_period"] = pd.PeriodIndex(monthly_perf["purchase_month"], freq="M")
monthly_perf = monthly_perf.set_index("_month_period").reindex(pd.period_range(first_purchase_date.to_period("M"), last_purchase_date.to_period("M"), freq="M"))
monthly_perf.index.name = "_month_period"
monthly_perf["purchase_month"] = monthly_perf.index.astype("string")
for count_column in ["orders_placed", "eligible_delivered_orders", "late_orders"]:
    monthly_perf[count_column] = monthly_perf[count_column].fillna(0).astype("int64")
monthly_perf["late_delivery_rate"] = monthly_perf["late_orders"] / monthly_perf["eligible_delivered_orders"].replace(0, np.nan)
monthly_perf["on_time_or_early_orders"] = monthly_perf["eligible_delivered_orders"] - monthly_perf["late_orders"]
monthly_perf["is_month_present_in_source"] = monthly_perf["orders_placed"].gt(0)
monthly_perf = monthly_perf.reset_index()
monthly_perf["is_partial_calendar_month"] = monthly_perf["purchase_month"].isin([first_month, last_month])

# Additional source-coverage diagnostic: flag a sharp volume collapse in the final six observed months.
# This is a warning heuristic, not proof that those months are incomplete.
monthly_perf["_month_period"] = pd.PeriodIndex(monthly_perf["purchase_month"], freq="M")
monthly_perf = monthly_perf.sort_values("_month_period").reset_index(drop=True)
monthly_perf["prior_3_month_median_orders"] = monthly_perf["orders_placed"].shift(1).rolling(3, min_periods=3).median()
last_period = monthly_perf["_month_period"].max()
last_six_months = monthly_perf["_month_period"].ge(last_period - 5)
volume_collapse = last_six_months & monthly_perf["prior_3_month_median_orders"].notna() & (monthly_perf["orders_placed"] < 0.5 * monthly_perf["prior_3_month_median_orders"])
if volume_collapse.any():
    low_volume_tail_start = monthly_perf.loc[volume_collapse, "_month_period"].min()
    monthly_perf["is_low_volume_source_tail"] = monthly_perf["_month_period"].ge(low_volume_tail_start)
else:
    low_volume_tail_start = None
    monthly_perf["is_low_volume_source_tail"] = False

actual_delivery_max = orders["order_delivered_customer_date"].max()
lead_series = pd.to_numeric(eligible.get("purchase_to_delivery_calendar_days", pd.Series(dtype=float)), errors="coerce").dropna()
p99_lead_days = float(lead_series.quantile(0.99)) if len(lead_series) else np.nan
if pd.notna(actual_delivery_max) and pd.notna(p99_lead_days):
    approximate_followup_cutoff_date = actual_delivery_max.normalize() - pd.Timedelta(days=int(np.ceil(p99_lead_days)))
    approximate_followup_cutoff = approximate_followup_cutoff_date.strftime("%Y-%m-%d")
    month_end_dates = monthly_perf["_month_period"].dt.end_time.dt.normalize()
    monthly_perf["has_approx_99pct_followup"] = month_end_dates.le(approximate_followup_cutoff_date)
else:
    approximate_followup_cutoff_date = None
    approximate_followup_cutoff = None
    monthly_perf["has_approx_99pct_followup"] = False
monthly_perf["meets_minimum_monthly_volume"] = monthly_perf["eligible_delivered_orders"].ge(500)
monthly_perf["include_in_default_trend"] = (
    (~monthly_perf["is_partial_calendar_month"])
    & (~monthly_perf["is_low_volume_source_tail"])
    & monthly_perf["has_approx_99pct_followup"]
    & monthly_perf["meets_minimum_monthly_volume"]
)
monthly_perf = monthly_perf.drop(columns=["_month_period"], errors="ignore")
monthly_perf.to_csv(ANALYSIS_OUTPUT_DIR / "monthly_delivery_performance.csv", index=False)
display(monthly_perf.tail(12))

plot_monthly = monthly_perf.loc[monthly_perf["include_in_default_trend"]].copy()
fig, ax = plt.subplots(figsize=(12, 5))
if not plot_monthly.empty:
    ax.plot(plot_monthly["purchase_month"], plot_monthly["late_delivery_rate"] * 100, marker="o", linewidth=1.8)
    ax.tick_params(axis="x", rotation=45)
else:
    ax.text(0.5, 0.5, "No mature full-coverage months passed the default trend filter.", ha="center", va="center", transform=ax.transAxes)
ax.set_title("Late delivery rate by purchase cohort")
ax.set_xlabel("Purchase month")
ax.set_ylabel("Late delivery rate (%)")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(CHART_OUTPUT_DIR / "monthly_late_delivery_rate.png", dpi=160, bbox_inches="tight")
plt.show()
print(f"Source purchase date coverage: {first_purchase_date.date()} to {last_purchase_date.date()}")
print(f"Approximate latest date with 99th-percentile delivery follow-up: {approximate_followup_cutoff}")
print(f"Low-volume source tail begins: {low_volume_tail_start if low_volume_tail_start is not None else 'not detected'}")

## 5. Segment investigation: customer state

Rank states by both performance gap and potential workload. We require at least 500 eligible delivered orders per state for the main comparison. This is a practical minimum sample-size rule for this portfolio, not a significance test.

In [ ]:
overall_late_rate = late_rate
state_rows = eligible.copy()
state_rows["customer_state"] = state_rows["customer_state"].fillna("unknown").astype("string")
state_perf = state_rows.groupby("customer_state", dropna=False).agg(
    eligible_orders=("order_id", "nunique"),
    late_orders=("is_late_delivery", "sum"),
    mean_order_review_score=("average_review_score", "mean"),
    reviewed_orders=("has_review_score", "sum"),
    low_rated_orders=("is_low_rated_order_calc", "sum"),
    median_delay_days_among_late=("delivery_delay_days", lambda s: s[s > 0].median()),
).reset_index()
state_perf["late_delivery_rate"] = state_perf["late_orders"] / state_perf["eligible_orders"].replace(0, np.nan)
state_perf["low_rating_share"] = state_perf["low_rated_orders"] / state_perf["reviewed_orders"].replace(0, np.nan)
state_perf["late_rate_gap_vs_overall_pp"] = (state_perf["late_delivery_rate"] - overall_late_rate) * 100
state_perf["excess_late_orders_vs_overall_rate"] = (state_perf["late_orders"] - state_perf["eligible_orders"] * overall_late_rate).clip(lower=0)
state_perf["meets_minimum_volume"] = state_perf["eligible_orders"].ge(500)
state_perf = state_perf.sort_values(["meets_minimum_volume", "excess_late_orders_vs_overall_rate", "late_delivery_rate"], ascending=[False, False, False])
state_perf.to_csv(ANALYSIS_OUTPUT_DIR / "state_delivery_performance.csv", index=False)
state_candidates = state_perf.loc[state_perf["meets_minimum_volume"]].copy()
display(state_candidates.head(12))

chart_states = state_candidates.sort_values("late_delivery_rate", ascending=False).head(12).sort_values("late_delivery_rate")
fig, ax = plt.subplots(figsize=(9, 6))
if not chart_states.empty:
    ax.barh(chart_states["customer_state"], chart_states["late_delivery_rate"] * 100)
    ax.axvline(overall_late_rate * 100, linestyle="--", label=f"Overall: {overall_late_rate:.1%}")
    ax.legend()
ax.set_title("States with highest late-delivery rates (minimum 500 eligible orders)")
ax.set_xlabel("Late delivery rate (%)")
ax.set_ylabel("Customer state")
fig.tight_layout()
fig.savefig(CHART_OUTPUT_DIR / "state_late_delivery_rate.png", dpi=160, bbox_inches="tight")
plt.show()

## 6. Segment investigation: product category and seller exposure

Orders with several categories or sellers appear in multiple segment rows. The analysis deduplicates each `order_id + category` and `order_id + seller` pair before aggregating, so a seller's duplicate item lines do not multiply that order's exposure. The same late order can still be exposed to more than one seller/category; this is not seller fault attribution.

In [ ]:
# Category performance: one row per order-category exposure.
category_col = "product_category_label"
item_categories = items[["order_id", category_col]].copy()
item_categories[category_col] = item_categories[category_col].fillna("unknown").astype("string")
item_categories = item_categories.drop_duplicates(["order_id", category_col])
order_outcome_cols = ["order_id", "delivery_metric_eligible", "is_late_delivery", "average_review_score", "has_review_score", "is_low_rated_order_calc", "delivery_delay_days"]
category_exposure = item_categories.merge(orders[order_outcome_cols], how="left", on="order_id", validate="many_to_one")
category_exposure = category_exposure.loc[category_exposure["delivery_metric_eligible"]].copy()
category_exposure["is_late_delivery"] = as_boolean(category_exposure["is_late_delivery"])
category_perf = category_exposure.groupby(category_col, dropna=False).agg(
    eligible_order_exposures=("order_id", "nunique"),
    late_order_exposures=("is_late_delivery", "sum"),
    mean_order_review_score=("average_review_score", "mean"),
    reviewed_order_exposures=("has_review_score", "sum"),
    low_rated_order_exposures=("is_low_rated_order_calc", "sum"),
    median_delay_days_among_late=("delivery_delay_days", lambda s: s[s > 0].median()),
).reset_index()
category_perf["late_delivery_rate"] = category_perf["late_order_exposures"] / category_perf["eligible_order_exposures"].replace(0, np.nan)
category_perf["low_rating_share"] = category_perf["low_rated_order_exposures"] / category_perf["reviewed_order_exposures"].replace(0, np.nan)
category_perf["late_rate_gap_vs_overall_pp"] = (category_perf["late_delivery_rate"] - overall_late_rate) * 100
category_perf["excess_late_order_exposures_vs_overall_rate"] = (category_perf["late_order_exposures"] - category_perf["eligible_order_exposures"] * overall_late_rate).clip(lower=0)
category_perf["meets_minimum_volume"] = category_perf["eligible_order_exposures"].ge(100)
category_perf = category_perf.sort_values(["meets_minimum_volume", "excess_late_order_exposures_vs_overall_rate", "late_delivery_rate"], ascending=[False, False, False])
category_perf.to_csv(ANALYSIS_OUTPUT_DIR / "category_delivery_performance.csv", index=False)
display(category_perf.loc[category_perf["meets_minimum_volume"]].head(15))

# Seller performance: one row per order-seller exposure. This is descriptive, not causal attribution.
item_sellers = items[["order_id", "seller_id"]].dropna(subset=["order_id", "seller_id"]).drop_duplicates(["order_id", "seller_id"])
seller_exposure = item_sellers.merge(orders[order_outcome_cols], how="left", on="order_id", validate="many_to_one")
seller_exposure = seller_exposure.loc[seller_exposure["delivery_metric_eligible"]].copy()
seller_exposure["is_late_delivery"] = as_boolean(seller_exposure["is_late_delivery"])
seller_perf = seller_exposure.groupby("seller_id", dropna=False).agg(
    eligible_order_exposures=("order_id", "nunique"),
    late_order_exposures=("is_late_delivery", "sum"),
    mean_order_review_score=("average_review_score", "mean"),
    reviewed_order_exposures=("has_review_score", "sum"),
    low_rated_order_exposures=("is_low_rated_order_calc", "sum"),
).reset_index()
seller_perf["late_delivery_rate"] = seller_perf["late_order_exposures"] / seller_perf["eligible_order_exposures"].replace(0, np.nan)
seller_perf["low_rating_share"] = seller_perf["low_rated_order_exposures"] / seller_perf["reviewed_order_exposures"].replace(0, np.nan)
seller_perf["late_rate_gap_vs_overall_pp"] = (seller_perf["late_delivery_rate"] - overall_late_rate) * 100
seller_perf["excess_late_order_exposures_vs_overall_rate"] = (seller_perf["late_order_exposures"] - seller_perf["eligible_order_exposures"] * overall_late_rate).clip(lower=0)
seller_perf["meets_minimum_volume"] = seller_perf["eligible_order_exposures"].ge(100)
seller_perf = seller_perf.sort_values(["meets_minimum_volume", "excess_late_order_exposures_vs_overall_rate", "late_delivery_rate"], ascending=[False, False, False])
seller_perf.to_csv(ANALYSIS_OUTPUT_DIR / "seller_delivery_performance.csv", index=False)
display(seller_perf.loc[seller_perf["meets_minimum_volume"]].head(15))

## 7. Fulfillment-stage diagnostics

The stage durations are measured in elapsed hours from timestamps. Invalid or reversed intervals were set to null upstream; this report shows how much of the eligible delivery population has usable values. Median and 90th percentile help show typical and long-tail delays.

In [ ]:
stage_specs = [
    ("Purchase to approval", "purchase_to_approval_hours"),
    ("Approval to carrier handoff", "approval_to_carrier_hours"),
    ("Carrier handoff to customer delivery", "carrier_to_delivery_hours"),
]
stage_rows = []
for label, column in stage_specs:
    if column not in orders.columns:
        stage_rows.append({"fulfillment_stage": label, "source_column": column, "valid_interval_count": 0, "coverage_of_eligible_delivered_orders": np.nan, "median_hours": np.nan, "p90_hours": np.nan, "interpretation": "Required stage field missing from model."})
        continue
    values = pd.to_numeric(eligible[column], errors="coerce")
    values = values.loc[values.notna() & values.ge(0)]
    stage_rows.append({
        "fulfillment_stage": label,
        "source_column": column,
        "valid_interval_count": int(values.count()),
        "coverage_of_eligible_delivered_orders": values.count() / len(eligible) if len(eligible) else np.nan,
        "median_hours": values.median(),
        "p90_hours": values.quantile(0.90),
        "interpretation": "Only valid non-negative intervals are summarized; coverage should be checked before comparing stages."
    })
stage_summary = pd.DataFrame(stage_rows)
stage_summary.to_csv(ANALYSIS_OUTPUT_DIR / "fulfillment_stage_summary.csv", index=False)
display(stage_summary)

# Order/metric population checks for downstream consumers.
quality_rows = [
    {"check": "order_model_one_row_per_order", "observed": int(len(orders)), "expected": int(orders["order_id"].nunique()), "status": "PASS" if len(orders) == orders["order_id"].nunique() else "FAIL"},
    {"check": "order_item_key_unique", "observed": int(items.duplicated(["order_id", "order_item_id"]).sum()), "expected": 0, "status": "PASS" if not items.duplicated(["order_id", "order_item_id"]).any() else "FAIL"},
    {"check": "eligible_orders_have_late_flag", "observed": int(eligible["is_late_delivery"].notna().sum()), "expected": int(len(eligible)), "status": "PASS" if eligible["is_late_delivery"].notna().all() else "FAIL"},
    {"check": "late_plus_on_time_equals_eligible", "observed": int(n_late + n_ontime), "expected": int(n_eligible), "status": "PASS" if n_late + n_ontime == n_eligible else "FAIL"},
    {"check": "late_rate_between_0_and_1", "observed": float(late_rate), "expected": "0 <= rate <= 1", "status": "PASS" if pd.notna(late_rate) and 0 <= late_rate <= 1 else "FAIL"},
    {"check": "review_score_range", "observed": f"min={orders['average_review_score'].min()}, max={orders['average_review_score'].max()}", "expected": "within 1 to 5 when present", "status": "PASS" if orders["average_review_score"].dropna().between(1,5).all() else "FAIL"},
]
analysis_checks = pd.DataFrame(quality_rows)
analysis_checks.to_csv(ANALYSIS_OUTPUT_DIR / "analysis_quality_checks.csv", index=False)
display(analysis_checks)
if analysis_checks["status"].ne("PASS").any():
    raise AssertionError("Business analysis quality checks failed:\n" + analysis_checks.loc[analysis_checks["status"].ne("PASS")].to_string(index=False))

## 8. Turn the analysis into investigation priorities

An above-average rate is not sufficient on its own. Rank segments by both the observed late rate and the number of late orders above what would be expected if the segment performed at the overall baseline. This is a descriptive prioritization aid—not an estimate of causal impact or guaranteed improvement.

In [ ]:
# Combine high-volume state and category candidates into one queue for follow-up.
priority_parts = []
if not state_candidates.empty:
    s = state_candidates.copy()
    s["segment_type"] = "Customer state"
    s["segment"] = s["customer_state"].astype("string")
    s["eligible_order_exposures"] = s["eligible_orders"]
    s["late_order_exposures"] = s["late_orders"]
    s["excess_late_order_exposures"] = s["excess_late_orders_vs_overall_rate"]
    s = s[["segment_type", "segment", "eligible_order_exposures", "late_order_exposures", "late_delivery_rate", "late_rate_gap_vs_overall_pp", "excess_late_order_exposures"]]
    priority_parts.append(s)
cat_candidates = category_perf.loc[category_perf["meets_minimum_volume"]].copy()
if not cat_candidates.empty:
    c = cat_candidates.copy()
    c["segment_type"] = "Product category"
    c["segment"] = c["product_category_label"].astype("string")
    c["excess_late_order_exposures"] = c["excess_late_order_exposures_vs_overall_rate"]
    c = c[["segment_type", "segment", "eligible_order_exposures", "late_order_exposures", "late_delivery_rate", "late_rate_gap_vs_overall_pp", "excess_late_order_exposures"]]
    priority_parts.append(c)
priority_queue = pd.concat(priority_parts, ignore_index=True) if priority_parts else pd.DataFrame()
if not priority_queue.empty:
    priority_queue = priority_queue.sort_values(["excess_late_order_exposures", "late_delivery_rate"], ascending=[False, False])
priority_queue.to_csv(ANALYSIS_OUTPUT_DIR / "recommendation_candidates.csv", index=False)
display(priority_queue.head(15))

# Create a concise, reproducible findings note using computed results.
top_state_rate = state_candidates.sort_values("late_delivery_rate", ascending=False).head(1)
top_state_excess = state_candidates.sort_values("excess_late_orders_vs_overall_rate", ascending=False).head(1)
top_category_rate = cat_candidates.sort_values("late_delivery_rate", ascending=False).head(1)

lines = [
    "# Business Findings — Delivery Reliability & Customer Experience",
    "",
    "## Baseline",
    f"- Orders in the order-level model: **{n_orders:,}**.",
    f"- Eligible delivered orders: **{n_eligible:,}**.",
    f"- Late delivered orders: **{n_late:,} ({late_rate:.2%})**.",
    f"- Mean order-level review score for late orders with scores: **{late_mean_score:.2f}/5**.",
    f"- Mean order-level review score for on-time/early orders with scores: **{ontime_mean_score:.2f}/5**.",
    f"- Low-rating share (order-level mean score <=2): **{late_low_share:.2%}** among reviewed late orders versus **{ontime_low_share:.2%}** among reviewed on-time/early orders.",
    "",
    "## Investigation candidates",
]
if not top_state_rate.empty:
    r = top_state_rate.iloc[0]
    lines.append(f"- Highest late rate among states with at least 500 eligible orders: **{r['customer_state']} ({r['late_delivery_rate']:.2%}, n={int(r['eligible_orders']):,})**.")
if not top_state_excess.empty:
    r = top_state_excess.iloc[0]
    lines.append(f"- Largest positive excess late-order count vs the overall baseline among qualified states: **{r['customer_state']} ({r['excess_late_orders_vs_overall_rate']:.0f} excess orders, n={int(r['eligible_orders']):,})**.")
if not top_category_rate.empty:
    r = top_category_rate.iloc[0]
    lines.append(f"- Highest late rate among categories with at least 100 eligible order exposures: **{r['product_category_label']} ({r['late_delivery_rate']:.2%}, n={int(r['eligible_order_exposures']):,})**.")
qualified_months = monthly_perf.loc[monthly_perf["include_in_default_trend"] & monthly_perf["late_delivery_rate"].notna()].copy()
if not qualified_months.empty:
    peak_month = qualified_months.loc[qualified_months["late_delivery_rate"].idxmax()]
    lower_month = qualified_months.loc[qualified_months["late_delivery_rate"].idxmin()]
    lines.append(
        f"- Monthly late delivery rates vary materially across eligible purchase cohorts: **{peak_month['purchase_month']} "
        f"({peak_month['late_delivery_rate']:.2%}, {int(peak_month['late_orders']):,}/{int(peak_month['eligible_delivered_orders']):,})** versus "
        f"**{lower_month['purchase_month']} ({lower_month['late_delivery_rate']:.2%}, {int(lower_month['late_orders']):,}/{int(lower_month['eligible_delivered_orders']):,})**. "
        "Validate the trend and investigate order mix and fulfillment-stage differences before attributing a cause."
    )
lines += [
    "",
    "## Proposed next action",
    "1. Investigate the states with the largest number of late orders above the overall baseline, not only the highest rate.",
    "2. For those regions, break down valid fulfillment-stage durations and compare product mix and seller exposure.",
    "3. Track late delivery rate and low-rating share after a defined operational intervention; do not treat this historical analysis as proof of causality.",
    "",
    "## Limitations",
    "- This is historical marketplace data, mainly from 2016–2018; it is not current operational performance.",
    "- Delivery KPIs are conditional on delivered orders with valid delivery dates. Orders still in progress or not delivered are not in that denominator.",
    "- The late-order/review association is unadjusted; order mix, geography, category, seller and other factors may differ.",
    "- State, category and seller results are exposure-based. One order can involve multiple categories or sellers, so segment counts are not additive.",
    "- The dataset lacks complete logistics event traces, carrier assignment, operating cost and profit measures.",
]
findings_path = ANALYSIS_OUTPUT_DIR / "findings_summary.md"
findings_path.write_text("\n".join(lines), encoding="utf-8")
print(f"Saved findings summary: {findings_path.relative_to(PROJECT_ROOT)}")
display(Markdown("\n".join(lines)))

## 9. Senior analyst interpretation and sign-off checklist

Before moving these numbers into Power BI:

- Confirm the validation report from Notebook 02 still shows all `PASS` results.
- Review `analysis_quality_checks.csv`; every check should be `PASS`.
- Read `findings_summary.md` and verify every statement against the tables.
- Use `recommendation_candidates.csv` as a shortlist for investigation, not a causal ranking.
- Use `monthly_delivery_performance.csv`'s maturity and partial-month flags when plotting trends.
- Add metric definitions to the dashboard and label the historical period clearly.
- Do not present item price or payment totals as profit.

### Outputs produced

- `outputs/business_analysis/executive_kpi_summary.csv`
- `outputs/business_analysis/delivery_review_comparison.csv`
- `outputs/business_analysis/monthly_delivery_performance.csv`
- `outputs/business_analysis/state_delivery_performance.csv`
- `outputs/business_analysis/category_delivery_performance.csv`
- `outputs/business_analysis/seller_delivery_performance.csv`
- `outputs/business_analysis/fulfillment_stage_summary.csv`
- `outputs/business_analysis/analysis_quality_checks.csv`
- `outputs/business_analysis/recommendation_candidates.csv`
- `outputs/business_analysis/findings_summary.md`
- `outputs/charts/review_score_by_delivery_outcome.png`
- `outputs/charts/monthly_late_delivery_rate.png`
- `outputs/charts/state_late_delivery_rate.png`

**Next stage:** use the validated KPI definitions and priority segments to design the Power BI executive overview and operational investigation pages.